In [ ]:
import findspark
findspark.init()
from pyspark.sql.session import SparkSession
from pyspark.sql.functions import * 

spark = SparkSession.builder.config("spark.driver.memory", "8g").getOrCreate()

In [ ]:
path = 'D:\\Data Self Learning\\Tran Hoang Long _ Data Analyst Course\\Data Engineer\\Projects\\Video Streaming\\data\\bronze\\20220401.json'

df = spark.read.json(path)
df.show()

In [ ]:
df = df.select("_source.*")
df.show()

In [ ]:
total_devices = df.select("Contract","Mac").groupBy("Contract").count()
total_devices = total_devices.withColumnRenamed('count','TotalDevices')
total_devices.show()

In [ ]:
df = df.withColumn("Type",
		   when((col("AppName") == 'CHANNEL') | (col("AppName") =='DSHD')| (col("AppName") =='KPLUS')| (col("AppName") =='KPlus'), "Truyền Hình")
		  .when((col("AppName") == 'VOD') | (col("AppName") =='FIMS_RES')| (col("AppName") =='BHD_RES')| 
				 (col("AppName") =='VOD_RES')| (col("AppName") =='FIMS')| (col("AppName") =='BHD')| (col("AppName") =='DANET'), "Phim Truyện")
		  .when((col("AppName") == 'RELAX'), "Giải Trí")
		  .when((col("AppName") == 'CHILD'), "Thiếu Nhi")
		  .when((col("AppName") == 'SPORT'), "Thể Thao")
		  .otherwise("Error"))
df.show()

In [ ]:
statistics = df.select('Contract','TotalDuration','Type').groupBy('Contract','Type').sum()
statistics = statistics.withColumnRenamed('sum(TotalDuration)','TotalDuration')
statistics = statistics.groupBy('Contract').pivot('Type').sum('TotalDuration').na.fill(0)
statistics.show()

In [ ]:
result = statistics.join(total_devices,'Contract','inner')
result.show()

In [ ]:
result = result.withColumn("Most_Watched", 
           when(greatest("Truyền Hình", "Phim Truyện", "Giải Trí", "Thiếu Nhi", "Thể Thao") == col("Truyền Hình"), "Truyền Hình")
           .when(greatest("Truyền Hình", "Phim Truyện", "Giải Trí", "Thiếu Nhi", "Thể Thao") == col("Phim Truyện"), "Phim Truyện")
           .when(greatest("Truyền Hình", "Phim Truyện", "Giải Trí", "Thiếu Nhi", "Thể Thao") == col("Giải Trí"), "Giải Trí")
           .when(greatest("Truyền Hình", "Phim Truyện", "Giải Trí", "Thiếu Nhi", "Thể Thao") == col("Thiếu Nhi"), "Thiếu Nhi")
           .when(greatest("Truyền Hình", "Phim Truyện", "Giải Trí", "Thiếu Nhi", "Thể Thao") == col("Thể Thao"), "Thể Thao")
          .otherwise("Error"))

result.show()

In [ ]:
result = result.withColumn("Customer_Taste",
            concat_ws(", ", 
            when(col("Giải Trí") > 0, "Giải Trí")
            .when(col("Thiếu Nhi") > 0, "Thiếu Nhi")
            .when(col("Thể Thao") > 0, "Thể Thao")
            .when(col("Truyền Hình") > 0, "Truyền Hình")
            .when(col("Phim Truyện") > 0, "Phim Truyện")
            .otherwise("Error")))

result.show()   